# Female Candidates

In [1]:
from importlib.metadata import version
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys


from scientisttools import FAMD, MCA
# Gower distance + UMAP — nonlinear mixed‑datatype embedding

# HDBSCAN — the best default for UMAP embeddings
import hdbscan
# Agglomerative (Hierarchical) Clustering
from sklearn.cluster import AgglomerativeClustering
from sklearn_extra.cluster import KMedoids

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# python source path
sys.path.append('../Src/')
# Set seed
SEED = 1776

# custom python
import plot
import utilities as u
import optimized_gower_distance

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Seaborn": sns.__version__,
    "Matplot": sys.modules['matplotlib'].__version__,
    "Sk-Learn": sys.modules['sklearn'].__version__,
    "Scipy": sys.modules['scipy'].__version__,
    "sklearn_extra": sys.modules['sklearn_extra'].__version__,
    "HDB Scan": version("hdbscan"),
    "Scientisttools": version("scientisttools"),
    "Gower": version("gower"),
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# display all columns
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Remove scientific notation
np.set_printoptions(suppress=True, precision=4, linewidth=100)
# reset options
# pd.reset_option('display.max_columns')

           Library  Version
0           Python  3.10.20
1           Pandas    2.3.3
2            NumPy   1.26.4
3          Seaborn   0.13.2
4          Matplot   3.10.8
5         Sk-Learn    1.7.2
6            Scipy   1.15.3
7    sklearn_extra    0.3.0
8         HDB Scan   0.8.42
9   Scientisttools    0.1.6
10           Gower    0.1.2


## Import Data

In [2]:
# import data
df = pd.read_parquet("../Data/Female_CAN_Heart_PreML.parquet")
mapping_df = pd.read_parquet("../Data/Female_CAN_Heart_Mapping.parquet")
# display shape
df.shape, mapping_df.shape

((8064, 112), (80, 18))

In [3]:
# check for NaNs
u.any_nans(df)

Clean Dataset: No NaNs found across 8,064 rows.


In [4]:
# display
df.head()

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,...,TotalBilirubinTransplant_Log,TotalBilirubinTransplant_Log_IsMissing,PC1_BMI,PC1_BMI_var,PC1_weight,PC1_weight_var,PC1_height,PC1_height_var,PC1_age,PC1_age_var
0,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_6,Group_3,...,0.405465,0,3.742386,1.0,2.043337,1.0,-2.418191,1.0,-1.670118,1.0
1,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_6,Group_4,...,0.470004,0,-1.218457,1.0,-2.587516,1.0,-4.307358,1.0,-0.641831,1.0
2,Group_1,Group_2,Other,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_4,...,0.262364,0,-0.981478,1.0,-1.075258,1.0,-0.445072,1.0,-0.127290,1.0
3,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_6,Group_4,...,0.336472,0,2.481018,1.0,1.531014,1.0,-1.396741,1.0,1.517571,1.0
4,Group_1,Group_2,Group_2,Group_2,Group_1,Group_1,Group_2,Group_1,Group_6,Group_3,...,0.875469,0,-0.575276,1.0,-0.345262,1.0,0.498493,1.0,0.386455,1.0


### User Function(s)

In [5]:
def display_mapping(mapping_data, feature):
    return mapping_data.loc[mapping_data.feature == feature].dropna(axis=1, how='all')

## Explortory Clustering

In [6]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8064 entries, 0 to 8063
Data columns (total 112 columns):
 #    Column                                        Non-Null Count  Dtype   
---   ------                                        --------------  -----   
 0    PreviousTransplantNumber_CAN                  8064 non-null   category
 1    WaitListDiagnosisCode_CAN                     8064 non-null   category
 2    Citizenship_CAN                               8064 non-null   category
 3    ResidencyStateRegistration_CAN                8064 non-null   category
 4    EducationLevel_CAN                            8064 non-null   category
 5    LifeSupportRegistration_ECMO_CAN              8064 non-null   category
 6    LifeSupportRegistration_IABP_CAN              8064 non-null   category
 7    LifeSupportMechanismRegistration_OTHER_CAN    8064 non-null   category
 8    VentricularDeviceBrandRegistration_CAN        8064 non-null   category
 9    FunctionalStatusRegistration_CAN       

In [7]:
# get columns
shadow_cols = df.columns[df.columns.str.contains('IsMissing')].tolist()
pca_cols =  df.columns[df.columns.str.contains('^PC')].tolist()
cat_cols = df.select_dtypes(include=["category"]).columns.tolist()
# all numeric cols & remove numeric columns above
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
num_cols = list(set(num_cols) - set(shadow_cols) - set(pca_cols))
num_cols.remove('TransplantSurvivalDay')
# remove label
df = df.drop('TransplantSurvivalDay', axis=1).copy()
# remove any unused categories
df = u.remove_cat_zero_count(df)
# sanity check
print(f"Shadow Columns: {len(shadow_cols)} PCA Columns: {len(pca_cols)} Numeric Columns: {len(num_cols)} Category Columns: {len(cat_cols)}")
print(f"Total Columns: {len(shadow_cols)+len(pca_cols)+len(cat_cols)+len(num_cols)} Data Frame Columns: {df.shape[1]}")

Shadow Columns: 6 PCA Columns: 22 Numeric Columns: 3 Category Columns: 80
Total Columns: 111 Data Frame Columns: 111


## Encode

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [9]:
pd.set_option('display.max_colwidth', None)  # Don't wrap text
pd.set_option('display.max_columns', None)    # Show all columns
df[cat_cols].sample(5)

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,PrimaryPaymentRegistration_CAN,DiagnosisAtListing_CAN,DiabetesType_CAN,DialysisTypeRegistration_CAN,DefibrillatorImplantRegistration_CAN,PriorCardiacSurgery_CAN,PriorCardiacSurgeryType_CAN,InitialWaitingListStatusCode_CAN,StatusAtTransplant_CAN,Ethnicity_CAN,VentilatorRegistration_CAN,WorkIncomeRegistration_CAN,AntigenBW4_CAN,AntigenBW6_CAN,AntigenC1_CAN,AntigenC2_CAN,AntigenDR51_CAN,AntigenDR51_2_CAN,AntigenDR52_CAN,AntigenDR52_2_CAN,AntigenDR53_CAN,AntigenDR53_2_CAN,AntigenDQ1_CAN,AntigenDQ2_CAN,FunctionalStatusTransplant_CAN,MedicalConditionTransplant_CAN,PrimaryPaymentTransplant_CAN,LifeSupportTransplant_ECMO_CAN,ResidencyStateTransplant_CAN,WorkIncomeTransplant_CAN,DialysisBetweenRegistrationTransplant_CAN,LifeSupportTransplant_IABP_CAN,InfectionTherapyIV_CAN,InotropesIVTransplant_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,SteroidsUse_CAN,TransfusionAfterRegistration_CAN,VentricularDeviceTypeTransplant_CAN,VentricularDeviceBrandTransplant_CAN,VentilatorySupport_CAN,VentilatorTransplant_CAN,PriorCardiacSurgeryTypeListAndTransplant_CAN,Hepatitis_B_CoreAntibody_CAN,SurfaceAntigenHEP_B_CAN,SurfaceHBVAntibodyTotalTransplant_CAN,CMVStatus_Transplant_CAN,HIV_SeroStatusTransplant_CAN,HEP_C_SerostatusStatus_CAN,EpsteinBarrSeroStatusTransplant_CAN,HIV_NAT_PreTransplant_CAN,HCV_NAT_PreTranspant_CAN,HBV_NAT_Result_CAN,PreviousTransplantSameOrgan_CAN,PreviousTransplantAnyOrgan_CAN,AntigenRA1_CAN,AntigenRA2_CAN,AntigenRB1_CAN,AntigenRB2_CAN,AntigenRDR1_CAN,AntigenRDR2_CAN,MalignancyBetweenRegistrationTransplant_CAN,CMV_IGG_Transplant_CAN,CMV_IGM_Transplant_CAN,PrimaryDiagnosisType_CAN,DialysisPriorRegistration_CAN,PriorCardiacSurgeryListAndTransplant_CAN,CrossMatchDone
5959,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_6,Group_2,Group_3,Group_3,Group_1,Group_2,Group_3,Group_1,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_2,Group_2,Group_1,Group_1,Group_2,Group_2,Group_5,Group_2,Group_3,Group_2,Group_3,Group_5,Group_5,Group_3,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_2,Group_4,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_3,Group_2,Group_2,Group_2,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_4,Group_3,Group_2,Group_3,Group_2,Group_2,Group_1,Group_1,Group_3
7796,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_3,Group_4,Group_2,Group_3,Group_1,Group_2,Group_3,Group_1,Group_1,Group_3,Group_2,Group_1,Group_1,Group_1,Group_2,Group_2,Group_1,Group_1,Group_2,Group_2,Group_5,Group_2,Group_3,Group_2,Group_3,Group_5,Group_6,Group_3,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_2,Group_4,Group_1,Group_1,Group_3,Group_1,Group_1,Group_4,Group_1,Group_1,Group_1,Group_3,Group_3,Group_3,Group_3,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_2,Group_4,Group_1,Group_1,Group_1,Group_2,Group_1,Group_2,Group_1
4165,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_6,Group_4,Group_3,Group_2,Group_1,Group_2,Group_3,Group_3,Group_1,Group_3,Group_1,Group_1,Group_1,Group_1,Group_2,Group_2,Group_1,Group_1,Group_2,Group_2,Group_5,Group_2,Group_3,Group_2,Group_3,Group_5,Group_1,Group_2,Group_3,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_2,Group_2,Group_2,Group_2,Group_1,Group_1,Group_1,Group_4,Group_1,Group_1,Group_3,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_3,Group_2,Group_2,Group_2,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_2,Group_4,Group_2,G

In [10]:
df[shadow_cols].sample(5)

,HemodynamicsTransplant_PCW_IsMissing,FunctionalStatusRegistration_IsMissing,FunctionalStatusTransplant_IsMissing,CreatinineRegistration_Log_IsMissing,CreatinineTransplant_Log_IsMissing,TotalBilirubinTransplant_Log_IsMissing
3659,0,0,0,0,0,0
6891,0,0,0,0,0,0
1714,0,0,0,0,0,0
193,0,0,0,0,0,0
6390,0,0,0,0,0,0


In [11]:
df[pca_cols].sample(5)

,PC1_hemodynamics,PC2_hemodynamics,PC3_hemodynamics,PC4_hemodynamics,PC5_hemodynamics,PC1_hemodynamics_var,PC2_hemodynamics_var,PC3_hemodynamics_var,PC4_hemodynamics_var,PC5_hemodynamics_var,PC1_creatinine_log,PC2_creatinine_log,PC1_creatinine_log_var,PC2_creatinine_log_var,PC1_BMI,PC1_BMI_var,PC1_weight,PC1_weight_var,PC1_height,PC1_height_var,PC1_age,PC1_age_var
2477,0.770888,-0.831421,2.838686,1.611131,0.002475,0.584419,0.166664,0.152436,0.051258,0.045223,9.366845,0.918844,0.847372,0.152628,1.012477,1.0,1.497687,1.0,1.301927,1.0,0.643626,1.0
7916,0.389451,-0.298686,-0.337016,0.044648,0.564469,0.584419,0.166664,0.152436,0.051258,0.045223,-0.526124,0.050058,0.847372,0.152628,-1.503763,1.0,-1.498609,1.0,-0.306459,1.0,1.209284,1.0
3873,0.992409,-1.433719,-1.594151,0.239514,1.140115,0.584419,0.166664,0.152436,0.051258,0.045223,0.076782,0.652964,0.847372,0.152628,-2.888630,1.0,-2.670058,1.0,-0.131020,1.0,-1.875775,1.0
2105,0.616808,-0.279577,-0.211135,-0.198869,-0.552272,0.584419,0.166664,0.152436,0.051258,0.045223,-0.866382,-0.410225,0.847372,0.152628,-0.679993,1.0,-1.025104,1.0,-1.027247,1.0,1.260598,1.0
7073,-2.504822,-0.028802,0.862270,-0.144053,-0.114047,0.584419,0.166664,0.152436,0.051258,0.045223,1.462531,-0.397375,0.847372,0.152628,0.200303,1.0,1.259440,1.0,2.590245,1.0,1.003427,1.0


In [12]:
df[num_cols].sample(5)

,DistanceFromDonorHospitaltoTXCenter_CAN,TotalBilirubinTransplant_Log,TotalDayWaitList_CAN
2382,369.0,0.405465,83.0
350,476.0,1.386294,11.0
408,278.0,0.182322,53.0
6296,382.0,0.470004,30.0
3139,40.0,0.182322,3.0


In [13]:
# instantiate
scaler = StandardScaler()

# Scale ONLY those columns
df[num_cols] = scaler.fit_transform(df[num_cols])
# display
df[num_cols].head()

,DistanceFromDonorHospitaltoTXCenter_CAN,TotalBilirubinTransplant_Log,TotalDayWaitList_CAN
0,-0.776794,-0.446057,-0.504504
1,-0.526761,-0.254880,-0.507482
2,-0.465349,-0.869953,3.513292
3,-0.886457,-0.650429,-0.233474
4,-0.930322,0.946196,-0.528330


In [14]:
df.head()

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,PrimaryPaymentRegistration_CAN,DiagnosisAtListing_CAN,DiabetesType_CAN,DialysisTypeRegistration_CAN,DefibrillatorImplantRegistration_CAN,PriorCardiacSurgery_CAN,PriorCardiacSurgeryType_CAN,InitialWaitingListStatusCode_CAN,TotalDayWaitList_CAN,StatusAtTransplant_CAN,Ethnicity_CAN,VentilatorRegistration_CAN,WorkIncomeRegistration_CAN,AntigenBW4_CAN,AntigenBW6_CAN,AntigenC1_CAN,AntigenC2_CAN,AntigenDR51_CAN,AntigenDR51_2_CAN,AntigenDR52_CAN,AntigenDR52_2_CAN,AntigenDR53_CAN,AntigenDR53_2_CAN,AntigenDQ1_CAN,AntigenDQ2_CAN,FunctionalStatusTransplant_CAN,MedicalConditionTransplant_CAN,PrimaryPaymentTransplant_CAN,LifeSupportTransplant_ECMO_CAN,ResidencyStateTransplant_CAN,WorkIncomeTransplant_CAN,DialysisBetweenRegistrationTransplant_CAN,LifeSupportTransplant_IABP_CAN,InfectionTherapyIV_CAN,InotropesIVTransplant_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,SteroidsUse_CAN,TransfusionAfterRegistration_CAN,VentricularDeviceTypeTransplant_CAN,VentricularDeviceBrandTransplant_CAN,VentilatorySupport_CAN,VentilatorTransplant_CAN,PriorCardiacSurgeryTypeListAndTransplant_CAN,Hepatitis_B_CoreAntibody_CAN,SurfaceAntigenHEP_B_CAN,SurfaceHBVAntibodyTotalTransplant_CAN,CMVStatus_Transplant_CAN,HIV_SeroStatusTransplant_CAN,HEP_C_SerostatusStatus_CAN,EpsteinBarrSeroStatusTransplant_CAN,HIV_NAT_PreTransplant_CAN,HCV_NAT_PreTranspant_CAN,HBV_NAT_Result_CAN,PreviousTransplantSameOrgan_CAN,PreviousTransplantAnyOrgan_CAN,AntigenRA1_CAN,AntigenRA2_CAN,AntigenRB1_CAN,AntigenRB2_CAN,AntigenRDR1_CAN,AntigenRDR2_CAN,MalignancyBetweenRegistrationTransplant_CAN,CMV_IGG_Transplant_CAN,CMV_IGM_Transplant_CAN,PrimaryDiagnosisType_CAN,DialysisPriorRegistration_CAN,PriorCardiacSurgeryListAndTransplant_CAN,DistanceFromDonorHospitaltoTXCenter_CAN,CrossMatchDone,HemodynamicsTransplant_PCW_IsMissing,PC1_hemodynamics,PC2_hemodynamics,PC3_hemodynamics,PC4_hemodynamics,PC5_hemodynamics,PC1_hemodynamics_var,PC2_hemodynamics_var,PC3_hemodynamics_var,PC4_hemodynamics_var,PC5_hemodynamics_var,FunctionalStatusRegistration_IsMissing,FunctionalStatusTransplant_IsMissing,CreatinineRegistration_Log_IsMissing,CreatinineTransplant_Log_IsMissing,PC1_creatinine_log,PC2_creatinine_log,PC1_creatinine_log_var,PC2_creatinine_log_var,TotalBilirubinTransplant_Log,TotalBilirubinTransplant_Log_IsMissing,PC1_BMI,PC1_BMI_var,PC1_weight,PC1_weight_var,PC1_height,PC1_height_var,PC1_age,PC1_age_var
0,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_6,Group_3,Group_2,Group_3,Group_1,Group_2,Group_3,Group_1,Group_1,Group_3,-0.504504,Group_2,Group_1,Group_1,Group_1,Group_2,Group_2,Group_1,Group_1,Group_2,Group_2,Group_5,Group_2,Group_3,Group_2,Group_3,Group_5,Group_3,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_2,Group_4,Group_1,Group_1,Group_1,Group_1,Group_1,Group_4,Group_1,Group_1,Group_1,Group_1,Group_3,Group_3,Group_3,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,Group_2,Group_4,Group_1,Group_1,Group_1,Group_2,Group_1,Group_1,-0.776794,Group_1,0,-2.474081,-0.208298,0.088947,-0.125106,0.069424,0.584419,0.166664,0.152436,0.051258,0.045223,0,0,0,0,0.509387,-0.015104,0.847372,0.152628,-0.446057,0,3.742386,1.0,2.043337,1.0,-2.418191,1.0,-1.670118,1.0
1,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_6,Group_4,Group_1,Group_2,Group_1,Group_2,Group_1,Group_1,Group_1,Group_3,-0.507482,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_1,Group_3,Group_2,Group_5,Group_2,Group_2,Group_2,Group_3,Group_4,Group_5,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,

In [15]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8064 entries, 0 to 8063
Data columns (total 111 columns):
 #    Column                                        Non-Null Count  Dtype   
---   ------                                        --------------  -----   
 0    PreviousTransplantNumber_CAN                  8064 non-null   category
 1    WaitListDiagnosisCode_CAN                     8064 non-null   category
 2    Citizenship_CAN                               8064 non-null   category
 3    ResidencyStateRegistration_CAN                8064 non-null   category
 4    EducationLevel_CAN                            8064 non-null   category
 5    LifeSupportRegistration_ECMO_CAN              8064 non-null   category
 6    LifeSupportRegistration_IABP_CAN              8064 non-null   category
 7    LifeSupportMechanismRegistration_OTHER_CAN    8064 non-null   category
 8    VentricularDeviceBrandRegistration_CAN        8064 non-null   category
 9    FunctionalStatusRegistration_CAN       

## Clustering
* [A modified and weighted Gower distance-based clustering analysis for mixed type data: a simulation and empirical analyses](https://pmc.ncbi.nlm.nih.gov/articles/PMC11654179/)
* [Distances with Mixed-Type Variables, some Modified Gower’s Coefficients](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://arxiv.org/pdf/2101.02481)